# Assignment 2
*Complete and hand in this completed worksheet (including its outputs and any supporting code outside of the worksheet) with your assignment submission.*

The assignment has three tasks:

- Task 1: Manually design features to characterize the kinematic and morphological aspects of acceleration time series. Subsequently, train a machine learning model, such as SVM or Random Forest, for to classify different activities.
- Task 2: Build and train a neural network to perform activity classification on the same dataset.
- Task 3: Fine-tune a pre-trained  neural network to perform activity classification on the same dataset.

The assignment has been modified from the [Deep Learning for Human Activity Recognition](https://github.com/mariusbock/dl-for-har/tree/main/tutorial_notebooks) tutorial presented at the 2021 ACM International Joint Conference on Pervasive and Ubiquitous Computing (UbiComp’21)/ International Symposium on Wearable Computers 21' (ISWC 21').

In [ ]:
# Run some setup code for this notebook
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import gc
import numpy as np
from tqdm import tqdm
from types import SimpleNamespace
from scipy.signal import butter, filtfilt, periodogram
import warnings
warnings.filterwarnings("ignore")

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

from cs690r.features import *
from cs690r.net_utils import *
from solutions_utils import Scaler3D

# The commands will allow the notebook to reload external python modules;
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

[Scikit-learn](https://scikit-learn.org/stable/index.html) is a Python package that provides simple and efficient tools for machine learning and data analysis. With a wide range of machine learning algorithms and evaluation metrics, it is widely used for building predictive models and conducting machine learning experiments.

To download and install the package, you can use the following command in their terminal or command prompt:

```bash
pip install -U scikit-learn
```
Refer to this [page](https://scikit-learn.org/stable/install.html) for more detailed information about scikit-learn installation. 

Once you install the scikit-learn, you can import tools for this assignment. Here are some examples you may find useful.

In [ ]:
# If you installed scikit-learn, you can run the following code
from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

In addtion to Scikit-learn, you will need to build and train neural networks for the activity classification task in this assignment. Common deep learning platforms include [PyTorch](https://pytorch.org/), [TensorFlow](https://www.tensorflow.org/), among others, which often require GPUs for efficient computation. In this assignment, the examples are given using PyTorch. But feel free to use other deep learning platforms that you are familiar with.

To install PyTorch, you can refer to this [page](https://pytorch.org/get-started/locally/) for detailed information.

If your laptop lacks GPU capabilities, you can leverage [Google Colab](https://colab.research.google.com/). Google Colab is a cloud-based platform provided by Google, offering a free Jupyter notebook environment integrated with Google Drive. It enables users to write and execute Python code within a browser-based interface, providing access to robust computing resources such as CPUs, GPUs, and TPUs. To use PyTorch in Google Colab, check out this [page](https://pytorch.org/tutorials/beginner/colab.html) for more information.

In [ ]:
# If you installed PyTorch, you can run the following code
import torch
import torch.nn as nn
import torch.optim as optim

# Task 1

In this assignment, we will use [RealWorld HAR dataset](https://www.uni-mannheim.de/dws/research/projects/activity-recognition/dataset/dataset-realworld/). This dataset captured 8 activities, including climbing stairs up and down, jumping, lying, running/jogging, sitting, standing, and walking, performed by 15 subjects. The original dataset contained acceleration, GPS, gyroscope, light, magnetic field, and sound level data. Sensors were positioned across multiple body locations, including the chest, forearm, head, shin, thigh, upper arm, and waist. Each subject engaged in each activity for approximately 10 minutes, except for jumping, which lasted around 1.7 minutes.

For simplicity, this assignment only focuses on wrist accelerometer data from 7 subjects pertaining to following five activities:
* walking
* running
* sitting
* standing
* climbing stairs up

Please note that Subject 2, Subject 4, and Subject 7 do not have climbing stairs up data. The data only contains acceleration captured by wrist sensors. The sampling rate is 50 Hz.

You can load the data using the provided code snippet. The data is structured as a dictionary, where the keys represent `(subject, activity)` pairs and the corresponding values denote the acceleration data from the wrist sensors.

In [ ]:
# Load activity data
data_path = os.path.join('data', 'activity_data.npz')
data = np.load(data_path, allow_pickle=True)
activity_data = data['data'].item()
del data_path, data

## **Task 1.1**

In this task, you'll create visualizations of the dataset to get a basic idea of how the acceleration time-series looks for each activity. The visualizations will help you understand the data better. Based on the visualizations, you can choose the right pre-processing methods and design features specific for this dataset for the classfication tasks.

1. Choose one subject's data randomly for each activity and use the `visualize_acceleration` function to plot the acceleration time-series. You may want to zoom in on the plot to see if there are any patterns within the acceleration time-series.

In [ ]:
def visualize_acceleration(acc, activity, subject):
    fig = plt.figure(figsize=(10, 3))
    ax = fig.add_subplot(111)
    ax.plot(acc)
    ax.set_xlabel('Frame')
    ax.set_ylabel('Acc ($m/s^2$)')
    ax.set_title('Subject {:d}, Activity: {:s}'.format(subject, activity))
    ax.legend(['X', 'Y', 'Z'])
    fig.tight_layout()
    plt.show()

In [ ]:
# Exercise 1.2 Visualize each activity
# TODO: Visualize walking


In [ ]:
# TODO: Visualize running


In [ ]:
# TODO: Visualize sitting


In [ ]:
# TODO: Visualize standing


In [ ]:
# TODO: Visualize climbing up


**Inline Question 1** Do you observe any patterns within each activity's acceleration data? If so, what are the patterns? Do they exhibit consistency across different subjects?

*Your Answers*

## **Task 1.2** 

1. Implement `low_pass_filter` below. In the function, you should 
    - Design a 6th order Butterworth low-pass filter with a cut-off frequency of 8 Hz.
    - Apply the low-pass filter to the input acceleration and return the filtered acceleration


2. After the implementation of the function, run the provided code to apply the low-pass filter to the entire dataset.

Feel free to explore additional preprocessing techniques based on your visualization results in addition to low-pass filtering.

In [ ]:
# Exercise 1.2.1 Implement `low_pass_filter`
def low_pass_filter(acc):
    """
    Low-pass filter the acceleration data.

    Parameters:
        acc (numpy.ndarray): Raw and unfiltered acceleration data.

    Returns:
        numpy.ndarray: Low-pass filtered acceleration data
    """
    ######################################################
    # TODO: Design a 6th order Butterworth low-pass filter
    # with a cut-off frequency of 8 Hz
    # Apply the low-pass filter to the input acc
    
    
    
    
    ######################################################
    
    return acc

Now, apply `low_pass_filter` to the accelerometer data. 

**Hint**

To check if your implementation of low-pass filter is correct, you can check the power spectral density (PSD) of the data before and after filtering. A correctly implemented filter should result in reduced power at frequencies higher than the cut-off frequency. Below is an example of how to use `scipy.signal.periodogram` to calculate PSD.

```python
# Load data
acc = activity_data[(1, 'climbingup')]

# Calculate the magnitude of the data
acc = np.linalg.norm(acc, axis=1)

# Calculate periodogram
f, Pxx = scipy.signal.periodogram(acc, fs=50)

# Plot frequency vs. PSD
fig = plt.figure(figsize=(7, 3))
ax = fig.add_subplot(111)
ax.plot(f, Pxx)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD ((m/s2) 2 / Hz)')
fig.tight_layout()
```

If your implementation of the low-pass filter is correct, you should obtain a plot similar to the example below. The power spectral density is significantly lower at frequencies above 8 Hz, indicating successful filtering by the low-pass filter with a cutoff frequency of 8 Hz.

<img src="imgs/psd.png">

In [ ]:
# Exercise 1.2.2 Apply `low_pass_filter`             
for key in activity_data:
    # Extract data
    acc = activity_data[key]
    
    # Apply the low-pass filter
    acc = low_pass_filter(acc)
    
    # Save
    activity_data[key] = acc
    
    del acc

## **Task 1.3** 

1. Implement `create_sliding_window` below. In the function, you should use sliding window technique to segment acceleration time-series. The default window size is set to 3 seconds (i.e. 150 data points), and the default step size is 1 second (i.e. 50 data points). If the length of the last time series is not sufficient to form a complete window, adjust it by including additional data points before it to form a complete window.

2. After the implementation of the function, run the provided code to segment the dataset into sliding windows.

In [ ]:
# Exercise 1.3.1 Implement `create_sliding_window`
def create_sliding_window(subject, activity, acc, wnd_sz=150, stp_sz=50):
    """
    Create sliding windows from the acceleration data.

    Parameters:
        subject (int): Subject identifier.
        activity (str): Activity label.
        acc (numpy.ndarray): Acceleration data.
        wnd_sz (int): Window size in data points. Default is 150.
        stp_sz (int): Step size in data points. Default is 50.

    Returns:
        windows: List of sliding windows, 
            each containing subject, activity, and acceleration data.
            
    """

    windows = []
    N = acc.shape[0]
    t = 0

    while t < N:
        #######################################
        # TODO: Use sliding windows to segment
        # the time series
        
        
        
        
        
        #######################################

        # Save
        wnd = SimpleNamespace()
        wnd.subject = subject
        wnd.activity = activity
        wnd.acc = acc[start_idx:end_idx]
        windows.append(wnd)

    return windows

Now, apply `create_sliding_window` to the accelerometer data. If your implementation is correct, the following code should yield the following message: *20827 windows were extracted*

In [ ]:
# Exercise 1.3.2 Apply `create_sliding_window`
windows = []
for subject, activity in activity_data:
    # Extract acceleration
    acc = activity_data[subject, activity]
    
    # Apply the sliding windows to the data
    windows.extend(create_sliding_window(subject, activity, acc))
windows = np.array(windows)
print('{:d} windows were extracted'.format(windows.shape[0]))

## **Task 1.4** 

In this task, you will extract features from each window for classification. The table below includes some features that are widely used to describe the characteristics of time-series, and some implementations of these features can be found in `features.py`. 

| Feature name                           | Description                                                            | How to use it                            |
|----------------------------------------|------------------------------------------------------------------------|--------------------------------|
| Mean                                   | Mean value of the time-series                                          | numpy.mean(x)                  |
| Standard deviation                     | Standard deviation of the time-series                                  | numpy.std(x)                   |
| Variance                               | Variance of the time-series                                            | numpy.variance(x)              |
| Median                                 | Median value of the time-series                                        | numpy.median(x)                |
| Mean absolute value                    | Mean absolute value of the time-series                                 | numpy.mean(np.abs(x))          |
| Root mean square                       | Root mean square of the time-series                                    | numpy.sqrt(numpy.mean(x ** 2)) |
| Coefficient of variation               | Ratio of the standard deviation to the mean                            | See `features.py`              |
| Interquartile range                    | Interquartile range of the time-series                                 | scipy.stats.iqr(x)             |
| Kurtosis                               | Kurtosis of the time-series                                            | scipy.stats.kurtosis(x)        |
| Skewness                               | Skewness of the time-series                                            | scipy.stats.skew(x)            |
| Number of peaks                        | The number of peaks of the time-series                                 | See `features.py`              |
| Number of crossings                    | The number of crossings of the time-series <br>on a specific number    | See `features.py`              |
| Dominant frequency                     | The frequency that has the most energy                                 | See `features.py`              |
| Energy ratio at the <br>dominant frequency | Ratio of the energy at the dominant <br>frequency to the sum of the energy | See `features.py`              |
| Total energy                           | Sum of the energy                                                      | See `features.py`              |
| Sample entropy                         | Quantify the complexity or regularity of <br>the time-series               | See `features.py`              |

1. Implement `extract_features_from_wnd` below. In this function, define features you consider necessary for the classification task and calculate them from the acceleration data in a sliding window for classification.

2. After the implementation of the function, run `extract_features` to build the complete feature set. Note that this process may take some time depending on the number of features generated. Consider saving the features locally for future use.

In [ ]:
# Exercise 1.4.1 Implement `extract_features_from_wnd`
def extract_features_from_wnd(acc):
    """
    Calculate features from the acceleration data.

    Parameters:
        acc (numpy.ndarray): Acceleration data.
            Shape: length * number of axes

    Returns:
        feats: List of feature values
        names: List of feature names
            
    """
    feats = []
    names = []
    _, axis = acc.shape
    axes = ['X', 'Y', 'Z']
    
    # mean 
    for i in range(axis):
        feats.append(np.mean(acc[:, i]))
        names.append('{:s}>mean'.format(axes[i]))
    
    ######################################################
    # TODO: Extract additional features that you deem relevant 
    # to the activities by referring to the table above.

    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    ######################################################
    
    return feats, names

In [ ]:
# Exercise 1.4.2 Apply `extract_features_from_wnd`
def extract_features(windows):
    """
    Extract features from the acceleration in the sliding windows

    Parameters:
        windows (list): The sliding windows.

    Returns:
        features (numpy.ndarray): features
        feature_names (numpy.ndarray): feature names
        activities (numpy.ndarray): 
        subjects (numpy.ndarray)
    """
    features = []
    feature_names = None
    activities = []
    subjects = []
    for wnd in tqdm(windows):
        feats, names = extract_features_from_wnd(wnd.acc)
        
        # save
        features.append(feats)
        if feature_names is None:
            feature_names = names
        activities.append(wnd.activity)
        subjects.append(wnd.subject)
        
        del feats, names
    return np.array(features), np.array(feature_names), np.array(activities), np.array(subjects)

features, feature_names, activities, subjects = extract_features(windows)

For convenient use in the future, consider saving your feature sets locally. You do not need to upload the features to Gradescope.

In [ ]:
output = {
    'features': features,
    'feature_names': feature_names,
    'activities': activities,
    'subjects': subjects
}
np.savez(os.path.join('solutions', 'features.npz'), **output)

To load features, use the following code.

In [ ]:
feature_dir = os.path.join('solutions', 'features.npz')
data = np.load(feature_dir, allow_pickle=True)
features = data['features']
feature_names = data['feature_names']
activities = data['activities']
subjects = data['subjects']
del data

## **Task 1.5** 

This task introduces you to the standard machine learning pipeline for HAR. Typically, it involves the following steps:

- **Training/Test split:** In this task, use Leave-One-Subject-Out cross validation (LOSOCV) for model training. Refer to [`sklearn.model_selection.LeaveOneGroupOut`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneGroupOut.html#sklearn.model_selection.LeaveOneGroupOut) for training/test split.
- **Normalization:** Normalize features to a similar range to prevent dominance by certain features. Explore data normalization methods in [scikit-learn's preprocessing module](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.preprocessing).
- **Feature selection:** Optimize model performance and reduce complexity by selecting relevant features. Explore feature selection methods in [scikit-learn's feature selection module](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.feature_selection). Note: Feature selection may not always be necessary, especially for models like Random Forest.
- **Hyper-parameter tuning:** Optimize hyperparameters using techniques like grid search or randomized search for maximum performance. Explore hyper-parameter optimization methods in [scikit-learn's hyper-parameter optimizer implementations](https://scikit-learn.org/stable/modules/classes.html#hyper-parameter-optimizers). Note that you should also use cross-validation to find the optimial parameters.
- **Train the model with optimal hyperparameters:**  Train the model on the training dataset using the optimized hyperparameters obtained in the previous step.
- **Model evaluation on the test set:** Evaluate the trained model's generalization performance on the test set to assess its effectiveness in making predictions on unseen data.

Here is the pseudocode you may find useful
```python
# Step 1: Split the data into training and test sets
for each training/test split in dataset:
    # Define training and test data
    X_train, y_train = training_set
    X_test, y_test = test_set
    
    # Step 2: Normalize the data
    # Fit the normalizer on the training set
    fit_normalizer(X_train)
    
    # Apply the normalizer to both training and test sets
    normalize(X_train)
    normalize(X_test)
    
    # Step 3: Feature selection
    # Choose a subset of features based on the training set
    fit_feature_selector(X_train)
    
    # Apply the feature selector to both training and test sets
    feature_selector(X_train)
    feature_selector(X_test)
    
    # Step 4: Hyper-parameter tuning
    # Identify important parameters and define value ranges
    # You can write the code in the following manner
    # or you can explore existing tools in scikit-learn
    scores = []
    for each param in param_list:
        for each inner training/validation split in training_set:
            inner_X_train, inner_y_train = inner_training_set
            inner_X_validation, inner_y_validation = inner_validation_set
            
            # Train the model on the inner training set
            model.fit(inner_X_train, inner_y_train)
            
            # Test the model on the inner validation set
            inner_pred = model.predict(inner_X_validation)
    
        # Aggregate the scores for all inner validation sets
        scores.append(inner_scores)
        
    # Choose the best parameter associated with the best score
    best_param = select_best_parameter(scores)
    
    # Step 5: Train the model on the training set using optimal parameters
    model = fit(X_train, y_train)
    
    # Step 6: Test the model on the test set
    # model.predict_proba() returns the probabilities of possible outcomes
    # model.predict() returns the predictions
    y_scores = model.predict_proba(X_test)
    y_pred = model.predict(X_test)
```

1. Run the provided code that uses `sklearn.preprocessing.LabelEncoder` to convert `activities` from strings into numerical labels.

2. Implement `train_model` to train the machine learning model in a LOSOCV manner.

3. Run the provided code to apply `train_model` on your feature set to perform classification.

In [ ]:
# Exercise 1.5.1: Convert activities into numerical labels
le = LabelEncoder().fit(activities)
labels = le.transform(activities)

In [ ]:
# Exercise 1.5.2: Implement `train_model'
def train_model(X, y, subjects):
    """
    Train machine learning model on in a LOSOCV manner

    Parameters:
        X (numpy.ndarray): Features
        y (numpy.ndarray): Labels
        subjects (numpy.ndarray): Subject identifiers

    Returns:
        - y_scores (numpy.ndarray): The predicted scores for each class for each sample.
        - y_pred (numpy.ndarray): The predicted labels for each sample.
    """
    n_samples, n_features = X.shape
    n_class = np.unique(y).size
    n_subjects = np.unique(subjects).size
    y_scores = np.zeros((n_samples, n_class))
    y_pred = np.zeros((n_samples, ))

    # The following code uses LeaveOneGroupOut to split the dataset
    # into training and test set
    for i, (train_mask, test_mask) in enumerate(LeaveOneGroupOut().split(X, y, subjects)):
        print('Training on subject {:d}/{:d}'.format(i+1, n_subjects), flush=True)
        
        # TODO: Use train_mask and test_mask to create
        # X_train, X_test: training and test features
        # y_train, y_test: training and test lables
        # groups_train, groups_test: training and test subjects
        
        
        
        
        # TODO: Choose a scaler from sklearn.preprocessing
        # Fit the scaler on training features
        # and apply the scaler to both training and test features
        
        
        
        
        # TODO: Choose a feature selection from sklearn.feature_selection
        # Fit the feature selector on training features
        # and apply the feature selector to both training and test features
        # This process should be done in a cross-validation manner

        
        
        
        # TODO: Hyper-parameter tuning
        # Define the parameters and the ranges of values you want to tune
        
        
        
        
        
        # TODO: Find the best parameters in a cross-valiation manner
        # You can write your own implementation, or use sklearn.model_selection
        # to find the best parameters
        # If using sklearn.model_selection, you can set parameters like this:
        # cv = 5 or cv = LeaveOneGroupOut()
        # scoring = 'f1_macro'
        
        
        
        
        
        
        
        # TODO: Use the best parameters to define the model
        # Name the model clf

        
        # TODO: Train the model by calling fit()

        
        # Test the model
        y_scores[test_mask] = clf.predict_proba(X_test)
        y_pred[test_mask] = clf.predict(X_test)
        
    return y_scores, y_pred

In [ ]:
# Exercise 1.5.3 Train the model
y_scores, y_pred = train_model(features, labels, subjects)

## **Task 1.6** 

1) Plot the confusion matrix of the classification results. If your model is successfully implemented, you may obtain a comparable confusion matrix as the following. However, the performance may vary depending on the types of features you have included and the model you have chosen.

<img src="imgs/exer1_confusion_matrix.png">

In [ ]:
# Exercise 1.6.1: Plot the confusiont matrix
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(labels, y_pred),
    display_labels=le.inverse_transform(np.unique(labels))
).plot(ax=ax)
fig.tight_layout()
plt.show()

2) Plot the ROC curve for each class. If your model is successfully implemented, you may obtain a comparable ROC curves as the following. However, the performance may vary depending on the types of features you have included and the model you have chosen.

<img src="imgs/exer1_roc.png">

In [ ]:
# Exercise 1.6.2 Plot the ROC curve
y_onehot_true = OneHotEncoder().fit_transform(labels.reshape(-1, 1)).toarray()
target_names = le.inverse_transform(np.unique(labels))

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)
for class_id in range(len(np.unique(labels))):
    fpr, tpr, _ = roc_curve(y_onehot_true[:, class_id], y_scores[:, class_id])
    ax.plot(fpr, tpr, label='ROC curve for {:s}, ROC-AUC: {:.2f}'.format(
        target_names[class_id], roc_auc_score(y_onehot_true[:, class_id], y_scores[:, class_id])))
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
fig.tight_layout()
plt.show()

# Task 2

In this task, you will create a simple Convolutional Neural Network (CNN) to perform classification on the acceleration time series. Unlike Task 1, there's no need for manual feature design as the CNN will learn relevant features from the data itself.

Run the provided code to check which device you are using for the model training.

In [ ]:
device = get_torch_device()
print('Using Device: ', device)

## **Task 2.1** 

1. Run the provided code to prepare the data for model training. The input acceleration time-series should be shaped into the format `(n_samples, n_channels, length)`, where `n_samples` is the number of sliding windows created, `n_channels` is the number of axis, and `length` is the length of the time-series within each sliding window (i.e. window size)

2. Run the provided code to create variables to hold the labels and subjects.

In [ ]:
# Exercise 2.1.1 Prepare the data
X = np.array([wnd.acc for wnd in windows])
X = np.swapaxes(X, 1, 2)

In [ ]:
# Exercise 2.1.2 Create variables to hold labels and subjects
activities = np.array([wnd.activity for wnd in windows])
le = LabelEncoder().fit(activities)
labels = le.transform(activities)

subjects = np.array([wnd.subject for wnd in windows])

## **Task 2.2**

1. Create a simple CNN. The architecture of the CNN is specified in the table below. You can use the default parameters unless specified.

| Layer                          | Settings                                                | Function                   |
|--------------------------------|---------------------------------------------------------|----------------------------|
| Convolutional layer            | input channels: 3, output channels: 30, kernel size: 15 | torch.nn.Conv1d            |
| Activation layer               | ReLU                                                    | torch.nn.Relu              |
| Batchnorm layer                | number of features: 30                                  | torch.nn.Batchnorm1d       |
| Max pooling layer              | kernel size: 2                                          | torch.nn.MaxPool1d         |
| Adaptive average pooling layer | output_size: 1                                          | torch.nn.AdaptiveAvgPool1d |
| Flatten                        | Root mean square of the time-series                     | torch.nn.Flatten           |
| Linear layer                   | input features: 30, output_features: 5                  | torch.nn.Linear            |
| Softmax layer                  | dimension: 1                                            | torch.nn.Softmax           |

**Hint**
You can look at the example provided on this [page](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) to learn how to use `torch.nn.Sequential` to define a neural network.

In [ ]:
# Exercise 2.2 Design the architecture of you deep learning model
class CNN(nn.Module):
    def __init__(self, input_size, output_size):
        super(CNN, self).__init__()
        
        self.model = nn.Sequential(
            ##################################
            # TODO: Define the network
            
            
            
            
            
            
            
            
            ##################################
        )
        
    def forward(self, X):
        return self.model(X)

## **Task 2.3**

In this task, you will fine-tune your CNN to achieve the best performance. The steps are similar to what you have done in **Task 1.5**.
- **Splitting the Dataset**: Divide the dataset into training, validation, and test sets. Make sure that the test set contains data from a single subject, with each subject represented in the test set exactly once. Split the remaining subjects into training and validation sets, ensure no overlap between sets.
- **Normalization**: Normalize the time series data to ensure consistent scaling.
- **Hyper-parameter Tuning**: Explore various optimization algorithms and their associated parameters, such as learning rate, to enhance the efficiency of neural network training. It's important to search for optimal parameter values, including learning rates, within a logarithmic space to ensure comprehensive exploration of the parameter space (see `numpy.logspace()`).
- **Training the Neural Network**: Train the model on the training dataset using the optimized hyperparameters obtained from the previous step.
- **Model Evaluation on the Test Set**: Evaluate the trained model's generalization performance on the test set to assess its effectiveness in making predictions on unseen data.

You may find the following pseudocode useful.

```python
# Step 1: Split the data into training/validation/test sets
for each training/validation/test split in dataset:
    # Define training, validation, and test data
    X_train, y_train = training_set
    X_val, y_val = validation_set
    X_test, y_test = test_set
    
    # Step 2: Normalize the data
    # Fit the normalizer on the training set
    fit_normalizer(X_train)
    
    # Apply the normalizer to both training and test sets
    normalize(X_train)
    normalize(X_val)
    normalize(X_test)
    
    # Step 3: Hyper-parameter tuning
    # Identify important parameters and define value ranges
    # You can write the code in the following manner
    # or you can explore existing tools in scikit-learn
    scores = []
    for each param in param_list:
        # Define the neural network model
        model = define_model()
        
        # Define the optimizer
        optimizer = define_optimizer(model, param)
        
        # Train the model on the training set
        train_model(model, optimizer, X_train, y_train)
            
        # Validate the mode on the validation set
        val_score = evaluate_model(model, X_val, y_val)
    
        # Save the score 
        scores.append(val_score)
        
    # Choose the best parameter associated with the best score
    best_param = select_best_parameter(scores)
    
    # Step 4: Train the model on the training set 
    # using the optimizer with the best parameters
    model = define_model()
    optimizer = define_optimizer(model, best_param)
    train_model(model, optimizer, X_train, y_train)
    
    # Step 5: Test the model on the test set
    y_scores = model(X_test)
    y_pred = predict_classes(y_scores)
```

1. Implement `train_cnn` and `train_cnn_helper` to fine-tune your CNN for optimal classification performance on the dataset.
2. Run the provided code to train your model.

Feel free to write your own function to fine-tune the model.

**Hint**

1. In `cs690r/net_utils.py`, we provided `train_model` and `eval_model`.
2. Turn to this [page](https://pytorch.org/docs/stable/optim.html) for how to use optimizer.

In [ ]:
def train_cnn_helper(device, n_class, X_train, X_val, y_train, y_val, epochs, batch_size):
    """
    Fine-tune the hyperparameters on a convolutional neural network (CNN).

    Args:
    - device (torch.device): The device to run the training process on (e.g., 'cuda' or 'cpu').
    - n_class (int): The number of classes for classification.
    - X_train (torch.Tensor): The training data features with shape (n_samples, n_channels, length).
    - X_val (torch.Tensor): The validation data features with shape (n_samples, n_channels, length).
    - y_train (torch.Tensor): The training data labels with shape (n_samples).
    - y_val (torch.Tensor): The validation data labels with shape (n_samples).
    - epochs (int): The number of epochs for training.
    - batch_size (int): The batch size for training.

    Returns:
    - best_model (torch.nn.Module): The best trained CNN model based on validation performance.
    """
    
    # To keep track of the best performance
    best_loss = np.inf
    best_model = None
    best_params = None
    
    input_size = X_train.shape[1]
    output_size = n_class
    
    # TODO: Define the range of values for the learning rate
    # Try numpy.logspace to explore learning rate in the logarithm space
    # Name the variable lr_attempts
    
    
    # Iterate each value for learning rate
    for lr in lr_attempts:
        
        # TODO: Define CNN model
        
        
        # TODO: Define optimizer
        
        
        # TODO: Define loss function
        # Use cross entropy as loss function
        
        
        # Train model
        # You should use train_model in the following way
        train_loss_history, val_loss_history = train_model(
            device, model, X_train, X_val, y_train, y_val,
            loss_func, optimizer, epochs=epochs, batch_size=batch_size
        )
        
        # TODO: Update best_loss, best_model, and
        # best_params based on the performance on the validation set
        
        
        
        
        
        del model, optimizer
        gc.collect()
    print('Best param: {:.3e}, best loss: {:.3e}'.format(best_params, best_loss), flush=True)
    return best_model

In [ ]:
def train_cnn(X, y, subjects, epochs=50, batch_size=32):
    """
    Train a Convolutional Neural Network (CNN) model for classification.

    Args:
    - X (numpy.ndarray): The input features with shape (n_samples, n_channels, length).
    - y (numpy.ndarray): The target labels with shape (n_samples).
    - subjects (numpy.ndarray): The subject IDs corresponding to each sample.
    - epochs (int): The number of epochs for training. Default is 50.
    - batch_size (int): The batch size for training. Default is 32.

    Returns:
    - y_scores (numpy.ndarray): The predicted scores for each class for each sample.
    - y_pred (numpy.ndarray): The predicted labels for each sample.
    """
    
    n_samples, n_channels, _ = X.shape
    n_class = np.unique(y).size
    n_subjects = np.unique(subjects).size
    
    y_scores = np.zeros((n_samples, n_class))
    y_pred = np.zeros((n_samples, ))
    
    # Use the following code to split data into training/validation/test set
    train_masks, val_masks, test_masks = determine_folds(subjects)
    for i, (train_mask, val_mask, test_mask) in enumerate(zip(train_masks, val_masks, test_masks)):
        print('Training on subject {:d}/{:d}'.format(i+1, n_subjects), end='\r', flush=True)
        
        # TODO: Use train_mask, val_mask, and test_mask to create
        # X_train, X_val, X_test: training, validation, and test data
        # y_train, y_val, y_test: training, validation, and test lables
        
        
        
        # TODO: Create your data normalization method
        # Fit the scaler on training data
        # and apply the scaler to training, validation, and test data
       
    
    
    
        
        # TODO: Convert X_train, X_val, X_test, y_train, and y_val
        # from numpy array to torch tensors
        # then move the data to the device for training
        
        
        
        
        
        
        
        # Find the best model by tuning hyperparameters
        best_model = train_cnn_helper(
            device, n_class, X_train, X_val, y_train, y_val,
            epochs=epochs, batch_size=batch_size
        )
        
        # Apply the best model on the test set
        scores = eval_model(device, best_model, X_test).detach().cpu().numpy()
        
        # Save your prediction results
        y_scores[test_mask] = scores
        y_pred[test_mask] = np.argmax(scores, axis=1)
    return y_scores, y_pred

In [ ]:
# Define your own epochs and batch_size
y_scores, y_pred = train_cnn(X, labels, subjects)

## **Task 2.4** 

1) Plot the confusion matrix of the classification results. 

2) Plot the ROC curve for each class.

If your model is successfully implemented, you may obtain a comparable confusion matrix and ROC curve as in Exercise 1.

In [ ]:
# Exercise 2.4.1 Plot the confusiont matrix
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(labels, y_pred),
    display_labels=le.inverse_transform(np.unique(labels))
).plot(ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
# Exercise 2.4.2 Plot the ROC curve
y_onehot_true = OneHotEncoder().fit_transform(labels.reshape(-1, 1)).toarray()
target_names = le.inverse_transform(np.unique(labels))

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)
for class_id in range(len(np.unique(labels))):
    fpr, tpr, _ = roc_curve(y_onehot_true[:, class_id], y_scores[:, class_id])
    ax.plot(fpr, tpr, label='ROC curve for {:s}, ROC-AUC: {:.2f}'.format(
        target_names[class_id], roc_auc_score(y_onehot_true[:, class_id], y_scores[:, class_id])))
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
fig.tight_layout()
plt.show()

**Inline Question 2** What are the advantages and disadvantages of each method, namely designing features and training a machine learning model, versus training a deep learning model?

*Your Answers*

# **Task 3**

In this task, you will leverage a pre-trained model and fine-tune it on this dataset to perform activity recognition. Pre-trained models are usually trained on large datasets and offer several advantages, such as improved generalization, faster convergence, and reduced data requirements for fine-tuning. By using a model that has already learned meaningful feature representations from a large dataset, we hope to achieve better performance with limited labeled data.

In particular, we are going to leverage a ResNet that was pre-trained on UK Biobank, a uniquely powerful biomedical database that contains a vast collection of activity recordings from a large and diverse population. This extensive dataset enables the model to learn rich representations of human movement patterns and physiological characteristics. You can find more information about how the ResNet was pre-trained in this [paper](https://www.nature.com/articles/s41746-024-01062-3) and [github repo](https://github.com/OxWearables/ssl-wearables).

By leveraging the pre-trained ResNet, we hope to enhance the accuracy and robustness of our activity recognition system, reduce training time, and improve generalization to unseen data, ultimately leading to more reliable and efficient activity classification.

Run the provided code to check which device you are using for the model training.

In [ ]:
device = get_torch_device()
print('Using Device: ', device)

## **Task 3.1**
Repeat **Task 2.1** and prepare the data for training.

In [ ]:
# Exercise 3.1.1 Prepare the data
GRAVITY_CONSTANT = 9.80665
X = np.array([wnd.acc for wnd in windows]) / GRAVITY_CONSTANT
X = np.swapaxes(X, 1, 2)

In [ ]:
# Exercise 3.1.2 Create variables to hold labels and subjects
activities = np.array([wnd.activity for wnd in windows])
le = LabelEncoder().fit(activities)
labels = le.transform(activities)

subjects = np.array([wnd.subject for wnd in windows])

## **Task 3.2** 
The ResNet model consists of two main parts: a **feature extractor** and a **classifier**. The feature extractor contains convolutional layers and batch normalization layers. These layers capture hierarchical spatial features from the input data. The weights for this part are loaded from the pre-trained model, which contains rich information on patterns and structures learned from a large dataset. The classifier consists of fully connected (FC) layers, which are primarily used to map the extracted features to specific activity classes for classification.

To fine-tune the model in this task, we are going to freeze the feature extractor, meaning that the weights in the convolutional layers will not be updated during training. This allows us to reuse the learned feature representations from the pre-trained model while only training the fully connected layers in the classifier. By doing so, we can reduce training time and improve generalization with a limited dataset.

1) Implement `freeze_weights`. In this function, freeze the weights related to feature extractor.
2) Run `setup_model` for later use.

**Hint**
1. To identify which weights belong to the feature extractor, you can inspect the parameters in the network using the following code snippet:

```python
for name, param in model.named_parameters():
    # check if name contains 'feature_extractor'
    pass
```

2. To freeze the weights, set `requires_grad = False`. If implemented correctly, a total of 63 weights should be frozen.
```python
param.requires_grad = False
```

In [ ]:
# Excercise 3.2.1 Implement `freeze_weights`
def freeze_weights(model):
    """
    Freeze the weights in the feature extractor of the model

    Args:
    - device (torch.device): The device to run the training process on (e.g., 'cuda' or 'cpu').
    """
    # # Used to count how many weights are frozen
    # # This is not necessary
    # i = 0
    
    #######################################
    # TODO: Freeze feature_extractor
    
    
    
    
    #######################################
    
    # print("Weights being frozen: %d" % i)
    model.apply(set_bn_eval)

In [ ]:
# Exercise 3.2.2 Run `setup_model`
def setup_model(device):
    """
    Set up the model for training, including define the model,
    load weights from pre-trained model, and freeze weights

    Args:
    - device (torch.nn.Module): The ResNet Model
    """
    
    # Define model
    model = make_model()
    
    # Load Pre-trained parameters
    weight_path = os.path.join('cs690r', 'mtl_5_best.mdl')
    load_weights(weight_path, model, device)
    
    # Freeze model
    freeze_weights(model)
    return model

## **Task 3.3**

In this task, you will fine-tune the ResNet to achieve the best performance. The steps are similar to what you have done in Task 2.3.

In [ ]:
def finetune_resnet_helper(device, n_class, X_train, X_val, y_train, y_val, epochs, batch_size):
    """
    Fine-tune the hyperparameters on a convolutional neural network (CNN).

    Args:
    - device (torch.device): The device to run the training process on (e.g., 'cuda' or 'cpu').
    - n_class (int): The number of classes for classification.
    - X_train (torch.Tensor): The training data features with shape (n_samples, n_channels, length).
    - X_val (torch.Tensor): The validation data features with shape (n_samples, n_channels, length).
    - y_train (torch.Tensor): The training data labels with shape (n_samples).
    - y_val (torch.Tensor): The validation data labels with shape (n_samples).
    - epochs (int): The number of epochs for training.
    - batch_size (int): The batch size for training.

    Returns:
    - best_model (torch.nn.Module): The best trained CNN model based on validation performance.
    """
    
    # To keep track of the best performance
    best_loss = np.inf
    best_model = None
    best_params = None
    
    input_size = X_train.shape[1]
    output_size = n_class
    
    # TODO: Define the range of values for the learning rate
    # Try numpy.logspace to explore learning rate in the logarithm space
    # Name the variable lr_attempts
    
    
    # Iterate each value for learning rate
    for lr in lr_attempts:
        
        # TODO: Set up model
        
        
        # TODO: Define optimizer

        
        # TODO: Define loss function
        # Use cross entropy as loss function
        
        
        # Train model
        # You should use train_model in the following way
        train_loss_history, val_loss_history = train_model(
            device, model, X_train, X_val, y_train, y_val,
            loss_func, optimizer, epochs=epochs, batch_size=batch_size
        )
        
        # TODO: Update best_loss, best_model, and
        # best_params based on the performance on the validation set
        
        
        
        
        
        del model, optimizer
        gc.collect()
    print('Best param: {:.3e}, best loss: {:.3e}'.format(best_params, best_loss), flush=True)
    return best_model

In [ ]:
def finetune_resnet(X, y, subjects, epochs=50, batch_size=32):
    """
    Train a Convolutional Neural Network (CNN) model for classification.

    Args:
    - X (numpy.ndarray): The input features with shape (n_samples, n_channels, length).
    - y (numpy.ndarray): The target labels with shape (n_samples).
    - subjects (numpy.ndarray): The subject IDs corresponding to each sample.
    - epochs (int): The number of epochs for training. Default is 50.
    - batch_size (int): The batch size for training. Default is 32.

    Returns:
    - y_scores (numpy.ndarray): The predicted scores for each class for each sample.
    - y_pred (numpy.ndarray): The predicted labels for each sample.
    """
    
    n_samples, n_channels, _ = X.shape
    n_class = np.unique(y).size
    n_subjects = np.unique(subjects).size
    
    y_scores = np.zeros((n_samples, n_class))
    y_pred = np.zeros((n_samples, ))
    
    # Use the following code to split data into training/validation/test set
    train_masks, val_masks, test_masks = determine_folds(subjects)
    for i, (train_mask, val_mask, test_mask) in enumerate(zip(train_masks, val_masks, test_masks)):
        print('Training on subject {:d}/{:d}'.format(i+1, n_subjects), end='\r', flush=True)
        
        # TODO: Use train_mask, val_mask, and test_mask to create
        # X_train, X_val, X_test: training, validation, and test data
        # y_train, y_val, y_test: training, validation, and test lables
        
        
        
        
        # TODO: Create your data normalization method
        # Fit the scaler on training data
        # and apply the scaler to training, validation, and test data
        
        
        
        
        # TODO: Convert X_train, X_val, X_test, y_train, and y_val
        # from numpy array to torch tensors
        # then move the data to the device for training
        
        
        
        
        
        
        
        # Find the best model by tuning hyperparameters
        best_model = finetune_resnet_helper(
            device, n_class, X_train, X_val, y_train, y_val,
            epochs=epochs, batch_size=batch_size
        )
        
        # Apply the best model on the test set
        scores = eval_model(device, best_model, X_test).detach().cpu().numpy()
        
        # Save your prediction results
        y_scores[test_mask] = scores
        y_pred[test_mask] = np.argmax(scores, axis=1)
    return y_scores, y_pred

In [ ]:
# Define your own epochs and batch_size
y_scores, y_pred = finetune_resnet(X, labels, subjects)

## **Task 3.4** 

1) Plot the confusion matrix of the classification results. 

2) Plot the ROC curve for each class.

If your model is successfully implemented, you may obtain a comparable confusion matrix and ROC curve as in Exercise 1. However, you may expect slightly worse results than those in the first two tasks.

In [ ]:
# Exercise 3.4.1 Plot the confusiont matrix
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(labels, y_pred),
    display_labels=le.inverse_transform(np.unique(labels))
).plot(ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
# Exercise 3.4.2 Plot the ROC curve
y_onehot_true = OneHotEncoder().fit_transform(labels.reshape(-1, 1)).toarray()
target_names = le.inverse_transform(np.unique(labels))

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)
for class_id in range(len(np.unique(labels))):
    fpr, tpr, _ = roc_curve(y_onehot_true[:, class_id], y_scores[:, class_id])
    ax.plot(fpr, tpr, label='ROC curve for {:s}, ROC-AUC: {:.2f}'.format(
        target_names[class_id], roc_auc_score(y_onehot_true[:, class_id], y_scores[:, class_id])))
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
fig.tight_layout()
plt.show()

**Inline Question 3** 
1.	What are the advantages and disadvantages of designing your own deep learning model versus fine-tuning a pre-trained model?
2.	If your results are better or worse than in Task 2, what do you think could be the reason?
*Your Answers*